In [1]:
import torch
import torch.nn as nn
import torch.onnx

# Example model (simple feedforward network)
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 2)
    
    def forward(self, x):
        return self.fc(x)

# Instantiate and set the model to evaluation mode
model = SimpleModel()
model.eval()


In [2]:
# Dummy input tensor (size must match your model's input)
dummy_input = torch.randn(1, 10)  # Batch size 1, input size 10

onnx_file = "simple_model.onnx"
torch.onnx.export(
    model,                      # PyTorch model
    dummy_input,                # Input tensor
    onnx_file,                  # Output file name
    export_params=True,         # Store trained weights
    opset_version=11,           # ONNX opset version (ensure compatibility)
    do_constant_folding=True,   # Optimize constant folding
    input_names=['input'],      # Name of the input
    output_names=['output'],    # Name of the output
    dynamic_axes={              # Dynamic axes for variable batch size
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model has been converted to ONNX and saved as {onnx_file}")

In [8]:
import torch
from train.agent.sac.actor import DiagGaussianActor

actor = DiagGaussianActor(obs_dim=28, action_dim=4, hidden_dim=64, hidden_depth=2,
                          log_std_bounds=[-5., 2.],lipsnet=False)  # hard coded for drone controllers.

actor.load_state_dict(torch.load('/home/haitong/PycharmProjects/sim_to_real/training/log/hover-aviary-v0/sac/sac_raw_force_input/1/log/best_actor.pth', map_location=torch.device('cpu')))

dummy_input = torch.randn(1, 28)  # Batch size 1, input size 10

In [9]:
onnx_file = "controller.onnx"
torch.onnx.export(
    actor.trunk,                      # PyTorch model
    dummy_input,                # Input tensor
    onnx_file,                  # Output file name
    export_params=True,         # Store trained weights
    opset_version=11,           # ONNX opset version (ensure compatibility)
    do_constant_folding=True,   # Optimize constant folding
    input_names=['input'],      # Name of the input
    output_names=['output'],    # Name of the output
    dynamic_axes={              # Dynamic axes for variable batch size
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)